# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access general metadata using properties
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields.

In [ ]:
# List all available record sets by their @id
print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"  - {record_set['@id']} (Name: {record_set.get('name', '(no name)')})")

# Let's pick the first record set for demonstration and list its fields by @id
# If multiple record sets are present, you can repeat for any others
record_sets = [r['@id'] for r in dataset.metadata.record_sets]
if record_sets:
    first_record_set_id = record_sets[0]
    record_set_metadata = [r for r in dataset.metadata.record_sets if r['@id'] == first_record_set_id][0]
    fields = record_set_metadata.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"\nFields for record set {first_record_set_id}:")
    for field in fields:
        print(f"  - {field['@id']} (Name: {field.get('name', '(no name)')})")
        # Optionally display columns inside field
        if 'column' in field:
            columns = field['column']
            if isinstance(columns, dict):
                columns = [columns]
            for col in columns:
                print(f"      * Column @id: {col['@id']}")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. All access is performed by `@id`.

In [ ]:
# Collect all record set @ids
record_sets = [r['@id'] for r in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_sets:
    # Load records for the record set by @id
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[record_set_id])} records for record set {record_set_id}")

# Display columns example for the first record set
if record_sets:
    print(f"\nColumns for record set {record_sets[0]}:")
    print(dataframes[record_sets[0]].columns.tolist())
    display(dataframes[record_sets[0]].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All column references are by `@id`.

In [ ]:
# Choose the first record set for demonstration
record_set_id = record_sets[0]
df = dataframes[record_set_id]

# List numeric fields by inspecting columns
print("All fields (by @id):")
for col in df.columns:
    print(f"- {col}")

# Example: Select a numeric field (edit as appropriate)
numeric_field_id = None
for col in df.columns:
    # Try to infer numeric (int/float) dtype by sample
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    raise ValueError("No numeric field found in the dataset.")

print(f"Selected numeric field for EDA: {numeric_field_id}")

# Filtering records: for demo, filter values > 10 (edit threshold)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (showing first 5):")
display(filtered_df.head())

# Normalization (Z-score)
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Grouping by a categorical field (pick first categorical, e.g. string, field)
group_field_id = None
for col in df.columns:
    if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped filtered data by {group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. All field references are by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field (filtered)
plt.figure(figsize=(7, 4))
sns.histplot(filtered_df[numeric_field_id], bins=15, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot of numeric field by group (if group field was determined)
if group_field_id:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.xticks(rotation=30)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion

In this notebook, we loaded a clinical dataset defined by a Croissant schema, inspected its record sets and fields by their `@id`, extracted records by `@id`, and conducted basic exploratory analysis and visualization. All exploration, filtering, and field selection was done explicitly via unique Croissant schema `@id` identifiers, ensuring transparent and reproducible data workflows with `mlcroissant`.

You can adjust the notebook (e.g., select different record sets, fields, thresholds, or grouping columns) by referencing their `@id` fields for deeper or alternate analysis.